# Speed-conditioned Weibull model of Time Headway (Weibull AFT)

Single distribution family (**Weibull**) for all 6 pairs (the two `*_2W` pairs are
dropped: the 2W leading class conflates motorised two-wheelers with bicycles).

**Model.** An Accelerated Failure Time (AFT) form of the Weibull, where the scale
depends on the follower's speed:

$$\text{headway} \sim \text{Weibull}(c,\ \lambda_i),\qquad \lambda_i = \exp(\beta_0 + \beta_1\,\text{speed}_i)$$

Shape \(c\) is shared within a pair; \(\beta_1\) is the speed effect. Fitted by
maximum likelihood with scipy (no external survival package, no matplotlib).

**What is tested.** For each pair, the speed model is compared with a speed-free
Weibull (same family) via a likelihood-ratio test and AIC. A significant, negative
\(\beta_1\) means higher follower speed compresses the headway distribution.

Speed is mean-centred for numerical stability; \(\beta_1\) is unaffected by centring.
Outputs (tables + native Excel charts) go to the `Tables` folder.

In [1]:
# --- Cell 1: Imports and paths ---
import os
import numpy as np
import pandas as pd
from scipy import stats, optimize
from openpyxl import load_workbook
from openpyxl.chart import ScatterChart, Reference, Series

BASE      = r"D:\Headway"
DATA_PATH = os.path.join(BASE, "data2.xlsx")
TABLES    = os.path.join(BASE, "Tables")
os.makedirs(TABLES, exist_ok=True)

In [2]:
# --- Cell 2: Load, drop the 2W pairs, rename working columns ---
df = pd.read_excel(DATA_PATH)
n_before = len(df)
df = df[df["V_Leading_Class"] != "2W"].copy()      # removes both *_2W pairs
df = df.rename(columns={"Time_Headway": "hw", "Target_Speed_km/hr": "spd"})

SMEAN = df["spd"].mean()                            # centring constant (km/h)
print(f"N before: {n_before}  ->  after dropping 2W: {len(df)}")
print("Pairs kept:", sorted(df['Pair'].unique()))
print("Mean subject speed (centring point): %.2f km/h" % SMEAN)

N before: 898  ->  after dropping 2W: 802
Pairs kept: ['BTW_following_4W', 'BTW_following_MT_3W', 'BTW_following_NMT_3W', 'PR_following_4W', 'PR_following_MT_3W', 'PR_following_NMT_3W']
Mean subject speed (centring point): 14.51 km/h


In [3]:
# --- Cell 3: Weibull AFT likelihood and fitters ---
def _nll_speed(p, t, x):
    """Neg log-lik: Weibull, shape=exp(p0), scale=exp(p1 + p2*x)."""
    c = np.exp(p[0])
    scale = np.exp(p[1] + p[2] * x)
    lp = stats.weibull_min.logpdf(t, c, loc=0, scale=scale)
    return -lp.sum() if np.all(np.isfinite(lp)) else 1e12

def _nll_null(p, t):
    """Neg log-lik: Weibull, shape=exp(p0), constant scale=exp(p1)."""
    c = np.exp(p[0])
    lp = stats.weibull_min.logpdf(t, c, loc=0, scale=np.exp(p[1]))
    return -lp.sum() if np.all(np.isfinite(lp)) else 1e12

def fit_pair(t, x):
    """Fit null (no speed) and speed models to one pair; return both results."""
    c0, _, s0 = stats.weibull_min.fit(t, floc=0)         # sensible init
    r0 = optimize.minimize(_nll_null, [np.log(c0), np.log(s0)], args=(t,),
                           method="Nelder-Mead", options={"maxiter": 8000})
    r1 = optimize.minimize(_nll_speed, [np.log(c0), np.log(s0), 0.0], args=(t, x),
                           method="Nelder-Mead",
                           options={"maxiter": 8000, "xatol": 1e-7, "fatol": 1e-7})
    return r0, r1

In [4]:
# --- Cell 4: Fit each pair; build results + predictions ---
per_rows, pred_rows, store = [], [], {}

for pair, g in df.groupby("Pair"):
    t = g["hw"].values
    s = g["spd"].values
    x = s - SMEAN
    n = len(t)

    r0, r1 = fit_pair(t, x)
    ll0, ll1 = -r0.fun, -r1.fun
    gamma, b0, b1 = r1.x
    c = np.exp(gamma)

    LR = 2 * (ll1 - ll0)
    p_lr = stats.chi2.sf(LR, 1)
    aic0, aic1 = 2 * 2 - 2 * ll0, 2 * 3 - 2 * ll1

    per_rows.append({
        "Pair": pair, "N": n, "shape_c": round(c, 3),
        "speed_coef_b1": round(b1, 4),
        "pct_scale_per_kmh":  round((np.exp(b1) - 1) * 100, 2),
        "pct_scale_per_5kmh": round((np.exp(5 * b1) - 1) * 100, 2),
        "LL_null": round(ll0, 2), "LL_speed": round(ll1, 2),
        "LR_chi2": round(LR, 2),
        "LR_p": "<0.001" if p_lr < 0.001 else round(p_lr, 4),
        "AIC_null": round(aic0, 2), "AIC_speed": round(aic1, 2),
        "dAIC_null_minus_speed": round(aic0 - aic1, 2),
        "speed_improves": "Yes" if p_lr < 0.05 else "No",
        "converged": bool(r0.success and r1.success),
    })

    # predicted median headway = scale * (ln2)^(1/c), at pair-specific speed quantiles
    q = np.percentile(s, [25, 50, 75])
    def median_at(sp):
        return np.exp(b0 + b1 * (sp - SMEAN)) * (np.log(2) ** (1 / c))
    pred_rows.append({
        "Pair": pair,
        "spd_p25": round(q[0], 1), "spd_p50": round(q[1], 1), "spd_p75": round(q[2], 1),
        "med_hw_p25": round(median_at(q[0]), 3),
        "med_hw_p50": round(median_at(q[1]), 3),
        "med_hw_p75": round(median_at(q[2]), 3),
    })
    store[pair] = (s, t, c, b0, b1)

per_pair   = pd.DataFrame(per_rows).sort_values("N", ascending=False).reset_index(drop=True)
predicted  = pd.DataFrame(pred_rows)
per_pair

,Pair,N,shape_c,speed_coef_b1,pct_scale_per_kmh,pct_scale_per_5kmh,LL_null,LL_speed,LR_chi2,LR_p,AIC_null,AIC_speed,dAIC_null_minus_speed,speed_improves,converged
0,BTW_following_4W,250,3.076,-0.0201,-1.99,-9.56,-310.26,-295.87,28.78,<0.001,624.53,597.74,26.78,Yes,True
1,BTW_following_MT_3W,186,2.524,-0.0320,-3.15,-14.77,-225.53,-207.40,36.26,<0.001,455.06,420.80,34.26,Yes,True
2,BTW_following_NMT_3W,160,2.346,-0.0214,-2.12,-10.14,-196.47,-192.29,8.37,0.0038,396.95,390.58,6.37,Yes,True
3,PR_following_MT_3W,104,3.840,-0.0301,-2.97,-13.99,-137.32,-128.28,18.08,<0.001,278.65,262.57,16.08,Yes,True
4,PR_following_NMT_3W,59,2.795,0.0047,0.48,2.40,-86.65,-86.58,0.14,0.7069,177.29,179.15,-1.86,No,True
5,PR_following_4W,43,3.156,-0.0013,-0.13,-0.63,-63.21,-63.21,0.01,0.9291,130.42,132.41,-1.99,No,True


In [5]:
# --- Cell 5: Predicted median headway across speed (shows compression) ---
predicted

,Pair,spd_p25,spd_p50,spd_p75,med_hw_p25,med_hw_p50,med_hw_p75
0,BTW_following_4W,12.0,14.9,19.9,2.437,2.295,2.079
1,BTW_following_MT_3W,12.2,15.7,21.1,2.063,1.840,1.548
2,BTW_following_NMT_3W,10.6,12.9,16.3,1.912,1.822,1.694
3,PR_following_4W,9.3,11.1,13.9,2.979,2.973,2.962
4,PR_following_MT_3W,9.1,10.8,13.2,3.058,2.909,2.704
5,PR_following_NMT_3W,8.9,10.7,12.6,2.629,2.652,2.676


In [6]:
# --- Cell 6: Pooled model - one overall test of the speed effect ---
# Weibull AFT with pair indicators (no global intercept) + shared shape;
# compared with and without subject speed.
pairs = sorted(df["Pair"].unique())
D = np.column_stack([(df["Pair"] == pp).values.astype(float) for pp in pairs])
xall = df["spd"].values - SMEAN
tall = df["hw"].values

def pooled_nll(p, with_speed):
    c = np.exp(p[0])
    inter = p[1:1 + len(pairs)]
    lin = D @ inter + (p[1 + len(pairs)] * xall if with_speed else 0.0)
    lp = stats.weibull_min.logpdf(tall, c, loc=0, scale=np.exp(lin))
    return -lp.sum() if np.all(np.isfinite(lp)) else 1e12

c0, _, s0 = stats.weibull_min.fit(tall, floc=0)
base = [np.log(c0)] + [np.log(s0)] * len(pairs)
rp0 = optimize.minimize(pooled_nll, base,          args=(False,),
                        method="Nelder-Mead", options={"maxiter": 20000, "fatol": 1e-6})
rp1 = optimize.minimize(pooled_nll, base + [0.0],  args=(True,),
                        method="Nelder-Mead", options={"maxiter": 20000, "fatol": 1e-6})
llp0, llp1 = -rp0.fun, -rp1.fun
LRp = 2 * (llp1 - llp0)
p_pool = stats.chi2.sf(LRp, 1)

pooled = pd.DataFrame([
    {"Model": "pair only",    "k": len(pairs) + 1, "LogLik": round(llp0, 2),
     "AIC": round(2 * (len(pairs) + 1) - 2 * llp0, 2)},
    {"Model": "pair + speed", "k": len(pairs) + 2, "LogLik": round(llp1, 2),
     "AIC": round(2 * (len(pairs) + 2) - 2 * llp1, 2)},
    {"Model": "LR test (df=1)", "k": None, "LogLik": round(LRp, 2),
     "AIC": "p<0.001" if p_pool < 0.001 else round(p_pool, 4)},
])
pooled

,Model,k,LogLik,AIC
0,pair only,7.0,-1034.69,2083.38
1,pair + speed,8.0,-995.54,2007.07
2,LR test (df=1),NaN,78.30,p<0.001


In [7]:
# --- Cell 7: Save tables to the Tables folder ---
out_path = os.path.join(TABLES, "headway_weibull_aft_speed.xlsx")
with pd.ExcelWriter(out_path, engine="openpyxl") as xl:
    per_pair.to_excel(xl,  sheet_name="AFT_per_pair",     index=False)
    predicted.to_excel(xl, sheet_name="Predicted_median", index=False)
    pooled.to_excel(xl,    sheet_name="Pooled_model",     index=False)
print("Saved:", out_path)

Saved: D:\Headway\Tables\headway_weibull_aft_speed.xlsx


In [8]:
# --- Cell 8: Per-pair chart: observed headway vs speed + fitted median line ---
wb = load_workbook(out_path)
for pair, (s, t, c, b0, b1) in store.items():
    sg = np.linspace(s.min(), s.max(), 80)
    mg = np.exp(b0 + b1 * (sg - SMEAN)) * (np.log(2) ** (1 / c))   # fitted median

    ws = wb.create_sheet(("m_" + pair)[:31].replace("/", "_"))
    ws["A1"], ws["B1"] = "obs_speed", "obs_headway"
    ws["D1"], ws["E1"] = "speed", "fitted_median_headway"
    for i in range(len(s)):
        ws.cell(row=i + 2, column=1, value=float(s[i]))
        ws.cell(row=i + 2, column=2, value=float(t[i]))
    for i in range(len(sg)):
        ws.cell(row=i + 2, column=4, value=float(sg[i]))
        ws.cell(row=i + 2, column=5, value=float(mg[i]))

    ch = ScatterChart()
    ch.title = f"{pair}: headway vs subject speed"
    ch.x_axis.title = "Subject speed (km/h)"
    ch.y_axis.title = "Time headway (s)"
    ch.x_axis.delete = False
    ch.y_axis.delete = False
    s_obs = Series(Reference(ws, min_col=2, min_row=1, max_row=len(s) + 1),
                   Reference(ws, min_col=1, min_row=2, max_row=len(s) + 1),
                   title_from_data=True)
    s_obs.marker.symbol = "circle"
    s_obs.graphicalProperties.line.noFill = True
    s_fit = Series(Reference(ws, min_col=5, min_row=1, max_row=len(sg) + 1),
                   Reference(ws, min_col=4, min_row=2, max_row=len(sg) + 1),
                   title_from_data=True)
    s_fit.smooth = True
    ch.series.append(s_obs)
    ch.series.append(s_fit)
    ch.height, ch.width = 9, 15
    ws.add_chart(ch, "G2")

wb.save(out_path)
print("Embedded per-pair charts into:", out_path)

Embedded per-pair charts into: D:\Headway\Tables\headway_weibull_aft_speed.xlsx
